# 07_deepeval_evaluate

07_deepeval_evaluate.py — DeepEval evaluate() 배치 평가

여러 테스트케이스 × 여러 메트릭. RAG end-to-end 평가의 표준 형태.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '07_deepeval_evaluate.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
07_deepeval_evaluate.py — DeepEval evaluate() 배치 평가

여러 테스트케이스 × 여러 메트릭. RAG end-to-end 평가의 표준 형태.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

from _common import banner, llm_unavailable
from _judges import deepeval_judge


def main() -> None:
    banner("DeepEval evaluate() — 3 케이스 × 2 메트릭 배치")
    judge = deepeval_judge()
    if judge is None:
        llm_unavailable()
        return

    test_cases = [
        LLMTestCase(
            input="에펠탑은 어디에 있나?",
            actual_output="에펠탑은 프랑스 파리에 있습니다.",
            expected_output="파리.",
            retrieval_context=["에펠탑은 프랑스 파리에 위치한다."],
        ),
        LLMTestCase(
            input="에펠탑은 언제 지어졌나?",
            actual_output="에펠탑은 1889년에 지어졌습니다.",
            expected_output="1889년.",
            retrieval_context=["에펠탑은 1889년 파리 만국박람회를 위해 세워졌다."],
        ),
        LLMTestCase(
            input="에펠탑의 높이는?",
            actual_output="에펠탑은 약 1000m 입니다.",  # 환각
            expected_output="약 330m.",
            retrieval_context=["에펠탑의 높이는 약 330m 이다."],
        ),
    ]

    metrics = [
        AnswerRelevancyMetric(threshold=0.7, model=judge),
        FaithfulnessMetric(threshold=0.7, model=judge),
    ]

    try:
        evaluate(test_cases=test_cases, metrics=metrics)
        # evaluate() 가 자체적으로 콘솔에 요약 출력
    except Exception as e:
        print(f"\n  ⚠ {type(e).__name__}: {str(e)[:300]}")


if __name__ == "__main__":
    main()


📌 DeepEval evaluate() — 3 케이스 × 2 메트릭 배치


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek/deepseek-v4-flash, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using deepseek/deepseek-v4-flash, strict=False, 
async_mode=True)...

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:              에펠탑의 높이는?                                                                     │
│  │     Actual Output:      에펠탑은 약 1000m 입니다.                                                            │
│  │     Expected Output:    약 330m.                                                                             │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy │ 1.00  │ 0.70      │ The score is 1.00 because the actual output dir...        │
│        FAIL  │ Faithfulness     │ 0.00  │ 0.70      │ The score is 0.00 because the actual output claims the    │
│              │                  │       │           │ Eiffel Tower height is about 1000m, but the retrieval     │
│              │                  │       │           │ context states it is about 330m, which is a direct        │
│              │                  │       │           │ contradiction.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Answer Relevancy                      │ 1.00                           │ 100.00%               │ 3             │
│  Faithfulness                          │ 0.67                           │ 66.67%                │ 3             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11426872;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 56.15s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.